# Co-positive triage of a trained model's high-loss groups

Runs the "is stage 2 worth building?" diagnostic from `REFERENCE_HAPLOTYPE_NOTES.md` ("Bootstrapping co-positives") for a
trained model M0 over one sample's image set, and splits the high-loss mass into:

| Category | Rule (g′ = real query's top alternative) | Meaning |
|---|---|---|
| a_lookalike | conf[g′] ≥ 0.5 | g′ cannot be separated from the label even with a *simulated* query: co-positive masking helps |
| a_semantic | separable, Truvari-equivalent to the sample's assembled paths | the label is the problem; g′ is arguably correct |
| b_flag | separable, not equivalent, p_real[g′] ≥ p_real[g\*] | label or simulation problem |
| c_model | otherwise | model uncertain on the real image |

`conf` and `sim_loss` need ≥ 2 simulated replicates per genotype. Instead of regenerating images with
`simulation.replicates=2`, this pairs **two image sets of the same regions** (e.g. `+repeat=0` and `+repeat=1`) generated
with a *deterministic* haplotype strategy (`TrueHaplotypeStrategy`), so candidate haplotypes, genotypes and labels match
and only the simulated reads differ (`copositive_pairs.py`). Replicate 0 is normally the set the model was trained on,
replicate 1 the held-out set (its real image is the query).

**Environment.** Run in the `npsv3-native` conda env with `module load npsv3/main` and jemalloc in `LD_PRELOAD` (see
`CLAUDE.md`), e.g. start the kernel/VS Code from a shell where those are set. Expensive steps are gated by flags and
cache their outputs in `WORKDIR`, so re-running the notebook only redoes the analysis:

1. **Record** (GPU, ~12 min for ~49k groups on one RTX A5000): submitted to SLURM when `RUN_RECORD=True` and the output is missing.
   Re-run the notebook once the job finishes.
2. **Semantic captures** (CPU, ~5 min per region, `CAPTURE_PARALLEL` at a time): only when `RUN_CAPTURES=True`; already
   captured regions are reused.

Steps follow the 2026-10-09 analysis of model `+train=0011` (HG00096); see "Co-positive triage results" in the notes.

In [ ]:
# ---- Parameters -------------------------------------------------------------------------------------------------
from pathlib import Path

REPO = Path("/home/mlinderman/research/npsv3-native")
SKILL = REPO / ".claude/skills/diagnose-kmer-evidence"
EXPERIMENTS = Path("/storage/mlinderman/projects/sv/npsv3-experiments/training")
RESOURCES = Path("/storage/mlinderman/projects/sv/npsv3-experiments/resources")
SAMPLE = "HG00096"

DATASET = "hgsvc3-hprc-2024-02-23-mc-chm13.pav.population.passing.training.hg38"
IMAGE_SUFFIX = ("graph.haplotype_strategy._target_=npsv3.images.haplotypes.TrueHaplotypeStrategy,"
                "graph.missing_are_ref=True,graph.population_prior=True,pileup=unphased_graph_kmer")
REP0 = EXPERIMENTS / f"{DATASET}.images/{SAMPLE}/+repeat=0,{IMAGE_SUFFIX}"  # trained-on simulations
REP1 = EXPERIMENTS / f"{DATASET}.images/{SAMPLE}/+repeat=1,{IMAGE_SUFFIX}"  # held-out simulations (+ real query)
CKPT = EXPERIMENTS / f"{DATASET}.models/+train=0011,data.batch_size=512/epoch=19-step=25980.ckpt"
# Optional: `command=loss` output for the same model and REP1, to check the recording reproduces it (None to skip)
LOSSES = EXPERIMENTS / f"{DATASET}.loss/{SAMPLE}/+train=0011,data.batch_size=512,sample={SAMPLE}/losses.pkl.gz"

# Outputs (must be on shared storage when recording via SLURM)
WORKDIR = EXPERIMENTS / f"test_outputs/copositive-0011-{SAMPLE}"

# Record step
RUN_RECORD = False  # submit the GPU recording job if WORKDIR/cop_r2.pkl.gz is missing
SLURM_ARGS = "--partition=gpu-short --gres=gpu:1 --cpus-per-task=8 --mem=48G --time=2:00:00"

# Analysis
CONF_THRESHOLD = 0.5  # conf[g'] at or above which g' is a look-alike
LOSS_QUANTILE = 0.9   # "high loss" cutoff
PIXEL_SAMPLE = 400    # groups for the image-space replicate-noise check (0 to skip; ~1 s/group on CPU)
FOCUS_REGIONS = ["chr4:109850259-109850582"]

# Semantic (Truvari) term on a random sample of high-loss b_flag groups where both g* and g' carry an SV
RUN_CAPTURES = False
N_SEMANTIC = 16
CAPTURE_PARALLEL = 4
SEED = 1
CAPTURE_OVERRIDES = [
    f"reference={RESOURCES}/Homo_sapiens_assembly38.fasta",
    f"input={RESOURCES}/hgsvc3-hprc-2024-02-23-mc-chm13/{DATASET}.vcf.gz",
    f"reads={RESOURCES}/sequence/{SAMPLE}.final.cram",
    "graph.ref_kmer_counts_kmc_prefix=" + str(RESOURCES) + "/Homo_sapiens_assembly38.non_unique.k${graph.kmer_size}",
    "graph.population_prior=True", "graph.missing_are_ref=True", "pileup=unphased_graph_kmer", "simulation.replicates=0",
    "graph.haplotype_strategy._target_=npsv3.images.haplotypes.TrueHaplotypeStrategy",
]
CAPTURE_SAMPLE_FIXTURE = "hg00096_sample"  # Sample fixture in tests/conftest.py

In [ ]:
import itertools
import os
import pickle
import shlex
import subprocess
import sys
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy.special import logsumexp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

sys.path.insert(0, str(SKILL))
import copositive as cp  # noqa: E402

WORKDIR.mkdir(parents=True, exist_ok=True)
RECORD = WORKDIR / "cop_r2.pkl.gz"


def n_haplotypes(G: int) -> int:
    return int((np.sqrt(8 * G + 1) - 1) / 2)


def genotypes(G: int) -> list[tuple[int, int]]:
    # Genotype i is the i-th pair of combinations_with_replacement(range(H), 2); haplotype 0 has no analysis-variant ALT
    return list(itertools.combinations_with_replacement(range(n_haplotypes(G)), 2))


def has_sv(g) -> bool:
    return any(h != 0 for h in g)

## 1. Record per-group diagnostics (GPU)

`copositive_pairs.py` encodes, per region, `[rep0 sims, rep1 sims, rep1 real]`, and calls `copositive.analyze_group` with
R=2. Regions whose metadata differ between the two sets are skipped and counted in the job log.

In [ ]:
if RECORD.exists():
    print(f"Using existing {RECORD}")
elif RUN_RECORD:
    script = WORKDIR / "record.sh"
    ld_preload = "/home/mlinderman/modules/jemalloc/5.3.0/install/lib/libjemalloc.so"
    script.write_text(f'''#!/bin/bash
#SBATCH {SLURM_ARGS} --job-name=copositive
source ~/.bashrc; conda activate npsv3-native; module load npsv3/main
case "$LD_PRELOAD" in *libjemalloc.so*) ;; *) export LD_PRELOAD={ld_preload}:$LD_PRELOAD;; esac
cd {REPO}
python3 {SKILL / "copositive_pairs.py"} --ckpt {shlex.quote(str(CKPT))} --rep0 {shlex.quote(str(REP0))} \\
    --rep1 {shlex.quote(str(REP1))} --output {RECORD} --workers 7
''')
    out = subprocess.run(["sbatch", "-o", str(WORKDIR / "record.%j.log"), str(script)], capture_output=True, text=True,
                         cwd=WORKDIR, check=True)
    print(out.stdout, "-> re-run the notebook once the job finishes")
else:
    raise SystemExit(f"{RECORD} missing; set RUN_RECORD=True to submit the recording job")

In [ ]:
df = pd.read_pickle(RECORD)
df["correct"] = df["prediction"] == df["genotype"]
print(f"{len(df)} groups, R={df.num_replicates.unique()}, real image differs between sets in "
      f"{(~df.real_same).mean():.1%} (random read subsampling at high depth)")
print(f"mean loss vs held-out sims {df.loss.mean():.3f}, vs trained-on sims {df.loss_rep0.mean():.3f}; "
      f"exact accuracy (R=2) {df.correct.mean():.3f}")

if LOSSES is not None and Path(LOSSES).exists():
    ref = pd.read_pickle(LOSSES)[["region", "loss"]]
    check = df[["region", "loss"]].merge(ref, on="region", suffixes=("", "_cmd"))
    print(f"rep1-only loss vs `command=loss`: {len(check)} regions matched, max |diff| "
          f"{(check.loss - check.loss_cmd).abs().max():.4f} (bf16/fp16 precision)")

## 2. Classify

Same rules as `copositive.py classify`. The b_flag rule here is `p_real[g′] ≥ p_real[g*]`, looser than the notes'
`> 0.8` flag rule, so section 6 splits b_flag by confidence.

In [ ]:
df = df.join(df.apply(lambda row: cp.classify_row(row, CONF_THRESHOLD), axis=1))
threshold = df.loss.quantile(LOSS_QUANTILE)
high = df[df.loss >= threshold]


def summary(frame):
    t = frame.groupby("category").agg(n=("loss", "size"), loss=("loss", "sum"), med_loss=("loss", "median"),
                                      med_sim_loss=("sim_loss", "median"), presence_differs=("presence_differs", "mean"))
    t.insert(1, "%groups", 100 * t.n / len(frame))
    t.insert(3, "%loss", 100 * t.loss / frame.loss.sum())
    return t.round(3)


print(f"All groups (total loss {df.loss.sum():.0f})")
display(summary(df))
print(f"High-loss groups: loss >= {threshold:.3f} (top {1 - LOSS_QUANTILE:.0%}), total loss {high.loss.sum():.0f}")
display(summary(high))

## 3. Is the loss at the look-alike floor?

If k negatives are truly indistinguishable from the label, InfoNCE cannot go below log(1 + kR). Real loss sitting *on*
the floor for every k means look-alikes explain the loss; well above it means something else does.

In [ ]:
floor = df.groupby("k_lookalike").agg(n=("loss", "size"), floor=("floor", "median"), real=("real_infonce", "median"),
                                      sim=("sim_loss", "median"))
floor["real - floor"] = floor.real - floor.floor
display(floor.round(3).T)

by_g = df.groupby("num_genotypes").agg(n=("loss", "size"), loss=("loss", "sum"), med_loss=("loss", "median"),
                                       acc=("correct", "mean"), med_k=("k_lookalike", "median"))
by_g["%loss"] = 100 * by_g.loss / df.loss.sum()
display(by_g.round(3))

## 4. What are the look-alikes?

* How close to identical: distribution of `conf` over all negatives.
* Haplotype-level equivalence: union haplotypes a, b when the homozygous genotypes (a,a) and (b,b) are look-alikes, then
  ask how many look-alike genotypes are explained by "same multiset of haplotype classes".
* Image space: is a look-alike's simulated image any further from the label's than two replicates of the label are?
  (mean |pixel difference| across replicates, per channel). Ratios near 1 mean the images themselves are
  indistinguishable, i.e. a data limit, not an encoder failure.

In [ ]:
negative_conf = np.concatenate([np.delete(r.conf, r.genotype) for r in df.itertuples()])
bins = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0001]
display(pd.Series(np.histogram(negative_conf, bins=bins)[0], index=pd.IntervalIndex.from_breaks(bins, closed="left"),
                  name="negatives by conf").to_frame().T)


def haplotype_classes(r, thr=CONF_THRESHOLD):
    G = r.num_genotypes
    gts = genotypes(G)
    H = n_haplotypes(G)
    index = {g: i for i, g in enumerate(gts)}
    C, diag = r.confusion, np.diag(r.confusion)
    conf = lambda i, j: 0.5 * (C[i, j] / max(diag[i], 1e-12) + C[j, i] / max(diag[j], 1e-12))  # noqa: E731
    parent = list(range(H))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for a, b in itertools.combinations(range(H), 2):
        if conf(index[(a, a)], index[(b, b)]) >= thr:
            parent[find(a)] = find(b)
    cls = [find(a) for a in range(H)]
    key = lambda g: tuple(sorted((cls[g[0]], cls[g[1]])))  # noqa: E731
    predicted = np.array([key(g) == key(gts[r.genotype]) for g in gts])
    observed = r.conf >= thr
    predicted[r.genotype] = observed[r.genotype] = False
    return pd.Series({"H": H, "n_classes": len(set(cls)), "k_obs": observed.sum(), "k_both": (predicted & observed).sum()})


big = df[df.num_genotypes >= 10]
classes = big.apply(haplotype_classes, axis=1)
print("Distinguishable haplotype classes by number of haplotypes (groups with G >= 10):")
display(pd.crosstab(classes.H, classes.n_classes))
print(f"look-alike genotypes explained by haplotype-class equivalence: {classes.k_both.sum() / classes.k_obs.sum():.1%}")

mispredicted = ~df.correct
lookalike_pred = df.apply(lambda r: r.conf[r.prediction] >= CONF_THRESHOLD, axis=1)
print(f"mispredictions whose prediction is a look-alike: {(lookalike_pred & mispredicted).sum() / mispredicted.sum():.1%}; "
      f"class-level accuracy (prediction in label + look-alikes) {(df.correct | lookalike_pred).mean():.3f} "
      f"vs exact {df.correct.mean():.3f}")

In [ ]:
if PIXEL_SAMPLE:
    from copositive_pairs import Pairs, read_meta, tar_index

    sample = df[(df.num_genotypes >= 10) & (df.k_lookalike > 0)].sample(PIXEL_SAMPLE, random_state=0)
    pairs = Pairs(list(sample.key), [read_meta(REP0), read_meta(REP1)], [tar_index(REP0), tar_index(REP1)])
    rows = []
    for i, r in enumerate(sample.itertuples()):
        _, images, _ = pairs[i]
        G, g = r.num_genotypes, r.genotype
        x = images[:-1].float().numpy().reshape(2, G, *images.shape[1:])  # (R, G, C, H, W)
        lookalike = [h for h in range(G) if h != g and r.conf[h] >= 0.9]
        separable = [h for h in range(G) if h != g and r.conf[h] < 0.1]
        cross = lambda a, b: 0.5 * (np.abs(x[0, a] - x[1, b]).mean(axis=(1, 2)) + np.abs(x[0, b] - x[1, a]).mean(axis=(1, 2)))  # noqa: E731
        same = cross(g, g)
        for name, others in (("lookalike", lookalike), ("separable", separable)):
            if others:
                rows.append(pd.DataFrame({"kind": name, "channel": range(x.shape[2]),
                                          "ratio": np.mean([cross(g, h) for h in others], axis=0) / same}))
    pixels = pd.concat(rows)
    print("Median ratio of cross-genotype to same-genotype (replicate noise) pixel difference, by channel:")
    display(pixels.groupby(["kind", "channel"]).ratio.median().unstack().round(3))

## 5. How does g′ relate to g\*?

Haplotype 0 carries no analysis-variant ALT allele (true for both haplotype strategies), so "SV presence" is "any
haplotype ≠ 0". Presence-changing look-alikes are *uncallable*, not equivalent: masking them would hide a real failure to
detect the SV.

In [ ]:
def relation(r):
    gts = genotypes(r.num_genotypes)
    a, b = gts[r.genotype], gts[r.alt]
    da, db = sum(h != 0 for h in a), sum(h != 0 for h in b)
    if da == 0:
        return "label no SV -> alt SV"
    if db == 0:
        return "label SV -> alt no SV"
    share = ", share hap" if set(a) & set(b) else ""
    return (f"dosage {da}->{db}" if da != db else "same dosage, other SV hap") + share


df["relation"] = df.apply(relation, axis=1)
for category in ("a_lookalike", "b_flag"):
    s = df[df.category == category]
    t = s.groupby("relation").agg(n=("loss", "size"), loss=("loss", "sum"), med_p_alt=("p_alt", "median"),
                                  med_conf=("conf_alt", "median"), med_sim_loss=("sim_loss", "median"))
    t["%loss"] = 100 * t.loss / s.loss.sum()
    print(category)
    display(t.sort_values("loss", ascending=False).round(3))


def presence_lookalikes(r):
    gts = genotypes(r.num_genotypes)
    label = has_sv(gts[r.genotype])
    return sum(1 for i, g in enumerate(gts) if i != r.genotype and r.conf[i] >= CONF_THRESHOLD and has_sv(g) != label)


df["presence_lookalikes"] = df.apply(presence_lookalikes, axis=1)
print(f"groups with a look-alike that changes SV presence: {(df.presence_lookalikes > 0).sum()} "
      f"({(df.presence_lookalikes > 0).mean():.1%})")

## 6. What would stage 2 remove, and what is left?

Soft negative masking: −log e^p / (e^p + Σ (1 − w_g) e^{s_g}) with w = conf (soft) or w = [conf ≥ threshold] (hard),
on the recorded R=2 real-query logits. This is an upper bound on what masking alone buys (a retrained model would move).

In [ ]:
def masked_infonce(r, hard=False):
    logits, g = r.query_logits.astype(float), r.genotype
    w = (r.conf >= CONF_THRESHOLD).astype(float) if hard else r.conf.copy()
    w[g] = 1.0
    negatives = np.log(np.clip(1 - w, 1e-12, None))[None, :] + logits
    negatives[:, g] = -np.inf
    positives = logits[:, g]
    return float(np.mean(np.logaddexp(positives, logsumexp(negatives)) - positives))


df["soft_masked"] = df.apply(masked_infonce, axis=1)
df["hard_masked"] = df.apply(lambda r: masked_infonce(r, hard=True), axis=1)
masked = df.groupby("category")[["real_infonce", "soft_masked", "hard_masked"]].sum()
masked.loc["total"] = masked.sum()
display(masked.round(0))

b = df[df.category == "b_flag"]
split = b.groupby(pd.cut(b.p_alt, [0, 0.5, 0.8, 1.0001], right=False)).agg(
    n=("loss", "size"), loss=("loss", "sum"), med_sim_loss=("sim_loss", "median"), presence_differs=("presence_differs", "mean"))
split["%all loss"] = 100 * split.loss / df.loss.sum()
print("b_flag by real-query confidence in g':")
display(split.round(3))
confident = b[b.p_alt >= 0.8]
print(f"confident flags: label no SV -> SV {(confident.genotype == 0).sum()}, "
      f"label SV -> no SV {((confident.genotype != 0) & (confident.alt == 0)).sum()}")

## 7. Semantic term on a sample of b_flag groups

Captures (`capture.py`, which wraps `true_haplotype_neighbors` for `TrueHaplotypeStrategy`) and `copositive.py semantic`
for `N_SEMANTIC` random high-loss b_flag groups where both g\* and g′ carry an SV (the ones Truvari equivalence could
reclassify), plus `FOCUS_REGIONS`. Captures are cached in `WORKDIR/caps/`.

In [ ]:
CAPS = WORKDIR / "caps"
CAPS.mkdir(exist_ok=True)
candidates = df[(df.category == "b_flag") & (df.loss >= threshold) & (df.num_genotypes >= 6)]
candidates = candidates[candidates.apply(lambda r: has_sv(genotypes(r.num_genotypes)[r.alt]) and
                                         has_sv(genotypes(r.num_genotypes)[r.genotype]), axis=1)]
semantic_regions = list(dict.fromkeys([*candidates.sample(min(N_SEMANTIC, len(candidates)), random_state=SEED).region,
                                       *FOCUS_REGIONS]))
print(f"{len(candidates)} eligible b_flag groups; sampling {len(semantic_regions)} regions")
cap_path = lambda region: CAPS / (region.replace(":", "_").replace("-", "_") + ".pkl")  # noqa: E731


def capture(region):
    env = {**os.environ, "REGION": region, "OUT": str(cap_path(region)),
           "SAMPLE_FIXTURE": CAPTURE_SAMPLE_FIXTURE, "CFG_OVERRIDES": shlex.join(CAPTURE_OVERRIDES)}
    cmd = (f"ulimit -v 64000000; exec python3 -m pytest {SKILL / 'capture.py'} -p tests.conftest --rootdir=. -q -x")
    with open(cap_path(region).with_suffix(".log"), "w") as log:
        return region, subprocess.run(["bash", "-c", cmd], cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT).returncode


missing = [region for region in semantic_regions if not cap_path(region).exists()]
if missing and RUN_CAPTURES:
    with ThreadPoolExecutor(CAPTURE_PARALLEL) as pool:
        for region, code in pool.map(capture, missing):
            print(region, "ok" if code == 0 else f"FAILED ({code}), see {cap_path(region).with_suffix('.log')}")
elif missing:
    print(f"{len(missing)} regions not captured; set RUN_CAPTURES=True")

In [ ]:
captured = [cap_path(region) for region in semantic_regions if cap_path(region).exists()]
SEMANTIC = WORKDIR / "sem_sample.pkl.gz"
if captured:
    subprocess.run([sys.executable, str(SKILL / "copositive.py"), "semantic", *map(str, captured), "--output", str(SEMANTIC)],
                   check=True, capture_output=True, env={**os.environ, "PYTHONWARNINGS": "ignore"})
    sem = pd.read_pickle(SEMANTIC).set_index("region")
    rows = []
    for region, s in sem.iterrows():
        r = df.set_index("region").loc[region]
        vector = np.asarray(s["sem"])
        if len(vector) != r.num_genotypes or s["sem_label"] != r.genotype:
            print(f"{region}: capture does not match the example (G {len(vector)} vs {r.num_genotypes}), skipped")
            continue
        with open(cap_path(region), "rb") as file:
            cap = pickle.load(file)
        gts, ref_len = genotypes(r.num_genotypes), len(cap["graph_ref_path"])
        size = lambda g: [len(cap["haps"][h]) - ref_len for h in gts[g]]  # noqa: E731
        rows.append({"region": region, "G": r.num_genotypes, "loss": r.loss, "label": f"{gts[r.genotype]} {size(r.genotype)}",
                     "alt": f"{gts[r.alt]} {size(r.alt)}", "p_alt": r.p_alt, "sem_alt": int(vector[r.alt]),
                     "n_equivalent": int(vector.sum()) - 1, "focus": region in FOCUS_REGIONS})
    semantic_table = pd.DataFrame(rows).set_index("region")
    display(semantic_table.round(3))
    sampled = semantic_table[~semantic_table.focus]
    print(f"sampled b_flag -> a_semantic: {sampled.sem_alt.sum()}/{len(sampled)} "
          "(label/alt show haplotype indices and length change vs hg38)")

## 8. Focus regions

In [ ]:
columns = ["genotype", "prediction", "alt", "category", "loss", "sim_loss", "k_lookalike", "conf_alt", "p_label", "p_alt"]
for region in FOCUS_REGIONS:
    row = df[df.region == region]
    if row.empty:
        print(f"{region}: not in the recorded groups")
        continue
    r = row.iloc[0]
    gts = genotypes(r.num_genotypes)
    display(row[columns].round(3))
    print(f"label {gts[r.genotype]}, prediction {gts[r.prediction]}; top real posteriors:",
          {gts[i]: round(float(r.p_real[i]), 3) for i in np.argsort(r.p_real)[::-1][:4]})

In [ ]:
df.drop(columns=["query_logits"]).to_pickle(WORKDIR / "cop_r2.final.pkl.gz")
high[high.category.isin(["a_lookalike", "b_flag"])].sort_values("loss", ascending=False)[
    ["region", "category", "loss", "sim_loss", "genotype", "alt", "conf_alt", "p_label", "p_alt"]
].to_csv(WORKDIR / "todo.tsv", sep="\t", index=False, float_format="%.4g")
print(f"Wrote {WORKDIR / 'cop_r2.final.pkl.gz'} and {WORKDIR / 'todo.tsv'}")